# Step 4. Inference and visualisation on one image

Loads the trained model, predicts vertebrae and corner points on a single radiograph, filters them (confidence 0.7, NMS IoU 0.3, keypoint score 0.9 or higher) and plots the original image, the detected points and the boxes side by side.

Code comments are in Spanish.

In [ ]:
# Instalar las bibliotecas necesarias
!pip install torch torchvision
!pip install opencv-python
!pip install numpy
!pip install gdown

# Importar las bibliotecas necesarias
import torch
from torchvision.models.detection import keypointrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import cv2
import numpy as np
from PIL import Image
from google.colab import drive
import gdown
from torchvision.ops import box_iou
from torchvision.ops import nms

In [ ]:
# Montar Drive de Google
drive.mount('/content/drive')

In [ ]:
# Definir las rutas
weights_path = 'MODELOV2' #Colocar ruta del modelo a utilizar para realizar predicciones
image_path = 'IMAGEN_PRUEBA' #Colocar ruta de imagen sobre la cual el modelo hara predicciones

In [ ]:
import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
# Descargar los pesos del modelo
#gdown.download('enlace_de_descarga_de_los_pesos_del_modelo', weights_path, quiet=False)

# Cargar el modelo preentrenado
def get_model(num_keypoints, weights_path):

    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   rpn_anchor_generator=anchor_generator,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes=2,  # El fondo es la primera clase, el objeto es la segunda clase
                                                                   pretrained_backbone=True)
    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)

    return model

model = get_model(num_keypoints=4, weights_path=weights_path)
model.eval()

In [ ]:
# Mover el modelo a una GPU si esta disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Cargar y pre-procesar la imagen
image = Image.open(image_path).convert("RGB")
image_tensor = F.to_tensor(image).unsqueeze(0).to(device)

# Inferencia
with torch.no_grad():
    prediction = model(image_tensor)

# Filtrar predicciones en base a umbral de confiabilidad y umbral de IoU
confidence_threshold = 0.7
iou_threshold = 0.3

boxes = prediction[0]['boxes'][prediction[0]['scores'] > confidence_threshold]
scores = prediction[0]['scores'][prediction[0]['scores'] > confidence_threshold]

# Aplicar supresion no-maxima utilizando box_iou
keep = nms(boxes, scores, iou_threshold)

In [ ]:
# Visualizar la imagen con Cuadros Delimitadores
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
for idx in keep:
    box = boxes[idx].cpu().numpy().astype(int)
    cv2.rectangle(image, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)

In [ ]:
# Convertir imagen a formato PIL para visualizar puntos clave
image_pil = Image.fromarray(image)

image_pil.show()

In [ ]:
# Impresiones de monitoreo
print("Numero de cuadros delimitadores antes de filtrar:", len(prediction[0]['boxes']))
print("Numero de cuadros delimitadores despues de filtrar:", len(keep))
print("Indices que permanecen despues del filtrado:", keep)

Number of bounding boxes before filtering: 29
Number of bounding boxes after filtering: 9
Indices to keep after filtering: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8], device='cuda:0')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import ImageDraw

# Cargar y preprocesar la imagen para mostrar
image = Image.open(image_path).convert("RGB")
image_array = np.array(image)

# Crear una figura y ejes
fig, ax = plt.subplots(1)

# Mostrar la imagen
ax.imshow(image_array)

# Mostrar cajas delimitadoras después de filtrar
for idx in keep:
    box = boxes[idx].cpu().numpy()
    rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)

# Mostrar puntos clave
keypoints = prediction[0]['keypoints'].cpu().numpy()
print("Forma del array de puntos clave:", keypoints.shape)  # Agregar esta línea para depuración

# Recopilar cajas delimitadoras finales y puntos clave
final_boxes = []
final_keypoints = []

# Mostrar puntos clave después de verificar las puntuaciones de confianza para las cajas delimitadoras finales
num_keypoints = 0
for idx in keep:
    keypoint_set = keypoints[idx]
    box = boxes[idx].cpu().numpy()

    final_boxes.append(box)  # Recopilar cajas delimitadoras finales

    for keypoint in keypoint_set:
        x, y, score = keypoint  # Extraer x, y y puntuación
        confidence_score = score.item() if isinstance(score, np.ndarray) else score

        # Verificar si la puntuación de confianza está dentro del rango deseado (por ejemplo, 0.9 a 1.0)
        if 0.9 <= confidence_score <= 1.0:
            num_keypoints += 1
            final_keypoints.append((x, y))  # Recopilar puntos clave finales

            # Agregar etiqueta con nivel de confianza a la caja delimitadora (tamaño de fuente más pequeño)
            label = f"{confidence_score:.2f}"
            ax.text(box[0], box[1], label, color='white', fontsize=5, bbox=dict(facecolor='red', alpha=0.8))

# Mostrar la imagen final
plt.axis('off')
plt.show()

print("Número de puntos clave dentro del rango de puntuación de confianza deseado para las cajas delimitadoras finales:", num_keypoints)

# Resultado de cajas delimitadoras finales y puntos clave
print("Cajas delimitadoras finales:", final_boxes)
print("Puntos clave finales:", final_keypoints)

Shape of keypoints array: (29, 4, 3)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
import numpy as np

# Cargar y preprocesar la imagen para mostrar
image = Image.open(image_path).convert("RGB")
image_array = np.array(image)

# Crear una figura y ejes con tres subgráficos
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Subgráfico 1: Imagen original
axs[0].imshow(image_array)
axs[0].set_title('Imagen Original')
axs[0].axis('off')

# Subgráfico 2: Imagen con puntos clave detectados
axs[1].imshow(image_array)
for point in final_keypoints:
    x, y = point
    axs[1].scatter(x, y, s=10, c='r')
axs[1].set_title('Imagen con Puntos Clave Detectados')
axs[1].axis('off')

# Subgráfico 3: Imagen con puntos clave detectados y cajas delimitadoras
axs[2].imshow(image_array)
for idx in keep:
    box = boxes[idx].cpu().numpy()
    rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor='r', facecolor='none')
    axs[2].add_patch(rect)

for point in final_keypoints:
    x, y = point
    axs[2].scatter(x, y, s=10, c='r')

# Ajustar los límites para mostrar todos los puntos clave
axs[2].set_xlim(0, image_array.shape[1])
axs[2].set_ylim(image_array.shape[0], 0)

axs[2].set_title('Imagen con Cajas Delimitadoras y Puntos Clave')
axs[2].axis('off')

# Mostrar las gráficas
plt.show()